In [1]:
from diffrax import diffeqsolve, Dopri5, ODETerm, SaveAt, PIDController, Kvaerno3, Tsit5
import diffrax
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
import time
import jax
jax.config.update("jax_enable_x64", True)

import sys
import os 
from tqdm import tqdm
from glob import glob
sys.path.append(os.getcwd()) 

import meanfield.hebbian_meanfield as mf
from importlib import reload

outdir = '/mnt/home/awakhloo/ceph/persistent-oscillations/results'
figdir = '/mnt/home/awakhloo/ceph/persistent-oscillations/figs'

In [ ]:
# np.random.seed(1223)
np.random.seed(8375)
# np.random.seed(23432)
g = 1.3 
N = 500
I = 0.55
# I= 5
f = 0.1 
k = 3.0 
p = 25
phi = jnp.tanh
omega = 2*np.pi*f

T = 100 
halt_time=0.5
halt = halt_time* T
dt_eval = 0.1
tvec_eval = np.linspace(0,T,int(T/dt_eval))

# random params
J = g/np.sqrt(N) * jnp.array(np.random.randn(N,N))
thetas = jnp.array(np.random.uniform(low=0,high=2*np.pi,size=(N,)))

x0 = np.random.randn(N)
A0 = np.zeros(N**2)
Y0 = jnp.concatenate([x0,A0])

sim_params = {
              'g' : g, 'I' : I, 'f' : f, 'k' : k, 'p' : p,  'omega' : omega,
              'T' : T, 'halt_time' : halt_time, 
              'J' : J, 'thetas' : thetas, 'halt' : T * halt_time,
             'Y0' : jnp.concatenate([x0,A0])}

out = mf.solve_rand_system(sim_params,T=T, Nsave=500, 
                          rtol=1e-6, atol=1e-6, 
                          dt_eval=dt_eval, N=N,
                              A_times =tvec_eval)
                         # A_times=[T*halt_time])
x, A = out 

In [ ]:
# fontsize
fn=22 
w, h = 8, 4.
lw=2.5
fig, ax = plt.subplots(1,2,figsize=(w,h)) 
# c1, c2 = 'orange', 'darkgreen'
c1,c2='darkgreen', 'darkgreen'
c3 = 'midnightblue'
c4=c3
alph=0.5
# i,j=0,1
i,j=15,10
phi=np.tanh
ax[0].plot(tvec_eval-T*halt_time, x[:,i], color=c1, lw=lw)  
ax[0].plot(tvec_eval-T*halt_time, x[:,j], color=c2, alpha=alph, lw=lw ) 

# ax[0].plot(tvec_eval-T*halt_time, phi(x)[:,i]) 
# ax[0].plot(tvec_eval-T*halt_time, phi(x)[:,j]) 

ax[1].plot(tvec_eval-T*halt_time, J[i,j]+A[:,i,j], color=c3, alpha=0.3, lw=lw) 
ax[1].plot(tvec_eval-T*halt_time, J[j,i]+A[:,j,i], color=c4,alpha=1.0, lw=lw)

for l in range(2): 
    # ax[l].set_xlabel("time from halt", fontsize=fn) 
    ax[l].set_xlabel("t", fontsize=fn)
    ylim=ax[l].get_ylim() 
    ax[l].vlines(0, ylim[0],ylim[1], color='grey', ls='--')
    ax[l].set_ylim(ylim) 
ax[0].set_ylabel("x(t)", fontsize=fn) 
ax[1].set_ylabel("J+A(t)",fontsize=fn)
ax[0].set_title("neuronal dynamics", fontsize=fn) 
ax[1].set_title("synaptic dynamics", fontsize=fn) 


# inset 
# inset Axes....
# x1, x2, y1, y2 = -10, 10, 0.006, 0.0068 # middle subregion of the original image
# x0,y0= 0.55, 0.2
x1, x2, y1, y2 = -50, -30, 0.0065, 0.007 # left subregion of the original image
x0,y0= 0.05, 0.2

ww=0.4

axins = ax[1].inset_axes(
    [x0, y0, ww, ww],
    xlim=(x1, x2), ylim=(y1, y2), 
    xticklabels=[x1,
                 (x1+x2)/2,
                 x2], 
    yticklabels=[y1,
                 round((y1+y2)/2,4),
                 y2])
axins.set_yticks([])
axins.set_xticks([])

axins.plot(tvec_eval-T*halt_time, J[i,j] + A[:,i,j], color=c3, alpha=0.3, lw=lw)
axins.plot(tvec_eval-T*halt_time, J[i,j]* np.ones(len(tvec_eval)), color='k', ls=':', lw=lw)

indc=ax[1].indicate_inset_zoom(axins, edgecolor="grey")
# indc.connectors(True, 

plt.tight_layout() 

fig.savefig(figdir + '/schematic_samples.png', dpi=300,
           bbox_inches='tight', transparent=False) 

In [ ]:
fig, axs = plt.subplots(3,1,figsize=(7.5,15))
T = np.linspace(0,4*np.pi,1000)
thetas = [0, np.pi,  np.pi/2-np.pi/6] 
for i, ax in enumerate(axs.flat):
    ax.plot(T, np.cos(T + thetas[i]), c='dimgrey', lw=8)
    ax.axis('off')
fig.savefig(figdir + '/oscillations_schematic.png', dpi=300, transparent=True,bbox_inches='tight')

In [36]:
for c in [c1,c2,c3,c4]: 
    print(matplotlib.colors.to_hex(c))

NameError: name 'matplotlib' is not defined

In [14]:
J[i,j]+A[:,i,j]

Array([0.00671572, 0.00671708, 0.00671728, 0.00671724, 0.00671762,
       0.00671872, 0.00672064, 0.00672333, 0.00672663, 0.00673038,
       0.00673438, 0.00673844, 0.00674239, 0.00674607, 0.00674935,
       0.00675211, 0.00675427, 0.00675575, 0.00675649, 0.00675647,
       0.00675566, 0.00675408, 0.00675173, 0.00674868, 0.00674497,
       0.00674069, 0.00673593, 0.00673079, 0.00672539, 0.00671983,
       0.00671423, 0.0067087 , 0.00670334, 0.00669825, 0.00669351,
       0.00668919, 0.00668534, 0.00668203, 0.00667929, 0.00667716,
       0.00667565, 0.00667479, 0.00667458, 0.00667503, 0.00667612,
       0.00667785, 0.0066802 , 0.00668312, 0.00668658, 0.00669053,
       0.00669494, 0.00669973, 0.00670486, 0.00671025, 0.00671585,
       0.0067216 , 0.00672744, 0.00673331, 0.00673915, 0.00674493,
       0.00675058, 0.00675606, 0.00676135, 0.00676638, 0.00677115,
       0.0067756 , 0.00677971, 0.00678347, 0.00678683, 0.00678978,
       0.00679231, 0.0067944 , 0.00679604, 0.00679724, 0.00679